# SST DIO: Minian parameter review, stage by stage (G27 HC2)

Plan: `plans/sst_dio_dreadd_validation_plan.md` §6.1. The pipeline is walked one stage at a time, as in the Minian notebook: each stage's variants are judged **at that stage's own output**, decided, and the decision is carried into the next stage. Claude's recommendation for each stage is in the plan, in bold. Kernel: `minian-native`.

Stages: 1 preprocessing · 2 seeds · 3 initialisation (all three decided: the pyramidal values, plus the lens mask) · 4 CNMF updates and merges. At the end: the one-at-a-time grid, kept as a sensitivity map.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.expanduser('~/code/caban_sst'))
sys.path.insert(0, os.path.expanduser('~/code/minian_vsekulic'))

import holoviews as hv
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image
from minian.visualization import CNMFViewer, VArrayViewer

from caban import sst_dio_parameter_review as review

%matplotlib inline
hv.notebook_extension('bokeh', width=100)
output_size = 100
STAGES_ROOT = os.path.expanduser('~/sst_dio_out/parameter_stages/G27_HC2')
GRID_ROOT = os.path.expanduser('~/sst_dio_out/parameter_grid/G27_HC2')
pd.read_csv(os.path.join(STAGES_ROOT, 'parameter_log.csv'))

## Stage 1: preprocessing (denoise `ksize` × background `wnd`) — decided: ksize 5, wnd 15

Measurements on the same 52 reference cells for every variant: `contrast_z`, `transient_snr`, `residual_background`, `core_fwhm_px` (see the plan).

In [ ]:
STAGE1 = os.path.join(STAGES_ROOT, 'preprocessing')
display(pd.read_csv(os.path.join(STAGE1, 'stage_metrics.csv'), index_col=0).round(3))
display(Image(os.path.join(STAGE1, 'stage_metrics.png')))
display(Image(os.path.join(STAGE1, 'max_proj_grid.png')))

In [ ]:
@widgets.interact(ksize=widgets.Dropdown(options=[3, 5, 7], value=5),
                  wnd=widgets.Dropdown(options=[10, 15, 20, 25, 30], value=20))
def show_preprocessing(ksize, wnd):
    review.compare_preprocessing(STAGE1, 'ksize_{}_wnd_{}'.format(ksize, wnd))
    plt.show()

In [ ]:
VARIANT = 'ksize_5_wnd_20'

Y_ref = review.open_stage_array(STAGE1, 'ksize_5_wnd_15', 'Y_fm_chk', intermediate=True)
Y_var = review.open_stage_array(STAGE1, VARIANT, 'Y_fm_chk', intermediate=True)
hv.output(size=int(output_size * 0.6))
vaviewer = VArrayViewer([Y_ref.rename('ksize_5_wnd_15'), Y_var.rename(VARIANT)],
                        framerate=5, summary=None, layout=True)
display(vaviewer.show())

## Stage 2: seeds (`max_wnd` × PNR threshold) — decided: max_wnd 15, PNR 1.0

From stage 1's decided movies, Minian's seed steps as the notebook runs them: initial seeds → inside the lens mask → PNR and KS → merged. Measured on the 52 reference cells: `recall` (cells with a final seed within 6 px), `seeds_per_matched_cell` and `cells_with_two_or_more` (split-cell risk), `unmatched_seeds` (seeds not on a reference cell: new cells or noise).

In [ ]:
STAGE2 = os.path.join(STAGES_ROOT, 'seeds')
display(pd.read_csv(os.path.join(STAGE2, 'stage_metrics.csv'), index_col=0).round(3))
display(Image(os.path.join(STAGE2, 'stage_metrics.png')))

### The seed steps of one variant, and its seeds not on a reference cell

Top: the seeds after each step. Below: every final seed not on a reference cell (red, numbered), each with a crop of the max projection and its trace with its PNR — a faint cell, or noise?

In [ ]:
@widgets.interact(max_wnd=widgets.Dropdown(options=[10, 15, 20], value=15),
                  pnr_thres=widgets.Dropdown(options=['0_75', '1_0', '1_25', '1_5'], value='1_0'))
def show_seeds(max_wnd, pnr_thres):
    name = 'max_wnd_{}_pnr_{}'.format(max_wnd, pnr_thres)
    display(Image(os.path.join(STAGE2, 'seed_steps_{}.png'.format(name))))
    display(Image(os.path.join(STAGE2, 'unmatched_{}.png'.format(name))))

In [ ]:
# review.log_decision(STAGES_ROOT, stage='seeds (max_wnd, PNR threshold)',
#                     values_tried='max_wnd 10/15/20 x PNR 0.75/1/1.25/1.5',
#                     value_chosen='max_wnd 15, PNR 1.0', why='...')

## Stage 3: initialisation and its merge — decided: initialise corr 0.8, wnd 10, merge 0.8

From the decided seeds, Minian's `initA` → `initC` → `unit_merge`. `recall`, `cells_with_two_or_more` (split), `fused_units` (one unit over two reference cells).

In [ ]:
STAGE3 = os.path.join(STAGES_ROOT, 'initialisation')
display(pd.read_csv(os.path.join(STAGE3, 'stage_metrics.csv'), index_col=0).round(3))
display(Image(os.path.join(STAGE3, 'stage_metrics.png')))

## Stage 4: CNMF — the merge after the first update × the spatial `dl_wnd`

Full pipeline runs with stages 1-3 as decided and the lens mask. Reference: `first_merge_0_8_dl_wnd_5` (the current values). Right panel: orange = reference only, blue = variant only, purple = both.

In [ ]:
STAGE4 = os.path.join(STAGES_ROOT, 'cnmf')
display(pd.read_csv(os.path.join(STAGE4, 'stage_metrics.csv'), index_col=0).round(3))
display(Image(os.path.join(STAGE4, 'stage_metrics.png')))

In [ ]:
finished4 = [name for name, r in review.variants(STAGE4).items()
             if r['status'] == 'done' and name != 'first_merge_0_8_dl_wnd_5']


@widgets.interact(variant=widgets.Dropdown(options=finished4))
def show_cnmf(variant):
    review.compare(STAGE4, variant, reference='first_merge_0_8_dl_wnd_5')
    plt.show()

### Minian's unit viewer on one stage 4 variant

Set `VARIANT`, run the cell: footprints, traces (C, S) and the motion-corrected movie.

**The units merge 0.7 would join** (unit ids in `first_merge_0_8_dl_wnd_5`, the default): A 6+7 (5.6 px apart, trace r 0.59), B 12+13 (9.1 px, r 0.55), C 35+36 (7.7 px, r 0.32), D 49+52+53 (7-10.6 px, r 0.47-0.69). One cell split, or neighbours?

In [ ]:
VARIANT = 'first_merge_0_8_dl_wnd_5'

A = review.open_array(STAGE4, VARIANT, 'A')
C = review.open_array(STAGE4, VARIANT, 'C')
S = review.open_array(STAGE4, VARIANT, 'S')
Y_fm_chk = review.open_array(STAGE4, VARIANT, 'Y_fm_chk', intermediate=True)
cnmfviewer = CNMFViewer(A=A, C=C, S=S, org=Y_fm_chk)
hv.output(size=int(output_size * 0.7))
display(cnmfviewer.show())

In [ ]:
# review.log_decision(STAGES_ROOT, stage='CNMF (first merge, dl_wnd)',
#                     values_tried='first merge 0.6/0.7/0.8 x dl_wnd 5/8',
#                     value_chosen='...', why='...')

## Sensitivity map: the one-at-a-time grid (full runs, final units)

Each variant changed one value from the current set and ran the whole pipeline; only the final units are shown, so a difference mixes the changed stage with everything after it. It shows which parameters matter at all. Right panel: orange = frozen only, blue = variant only, purple = both.

In [ ]:
display(review.summary_table(GRID_ROOT))
finished = [name for name, r in review.variants(GRID_ROOT).items()
            if r['status'] == 'done' and name != 'frozen']


@widgets.interact(variant=widgets.Dropdown(options=finished))
def show(variant):
    review.compare(GRID_ROOT, variant)
    plt.show()

In [ ]:
VARIANT = 'frozen'

A = review.open_array(GRID_ROOT, VARIANT, 'A')
C = review.open_array(GRID_ROOT, VARIANT, 'C')
S = review.open_array(GRID_ROOT, VARIANT, 'S')
Y_fm_chk = review.open_array(GRID_ROOT, VARIANT, 'Y_fm_chk', intermediate=True)
cnmfviewer = CNMFViewer(A=A, C=C, S=S, org=Y_fm_chk)
hv.output(size=int(output_size * 0.7))
display(cnmfviewer.show())